# Estudo ML: Stop Loss Dinâmico via ADX

**Objetivo:** Simular SL dinâmico baseado em ADX usando dados reais de 7 dias (9,685 candles 1m BTC-USD) e comparar com SL fixo atual (-0.20%).

**Hipótese:** SL adaptativo ao regime de volatilidade (ADX) reduz perdas em mercados fracos e permite respirar em mercados fortes, melhorando Win Rate e Sharpe.

## Tabelas Propostas

| ADX | SL Dinâmico | TP Dinâmico | R:R Ratio |
|-----|------------|-------------|----------|
| < 20 | 0.15% | 0.30% | 2.0:1 |
| 20-30 | 0.20% | 0.40% | 2.0:1 |
| 30-40 | 0.30% | 0.50% | 1.7:1 |
| > 40 | 0.40% | 0.60% | 1.5:1 |

In [1]:
import pandas as pd
import numpy as np
from decimal import Decimal
from pathlib import Path
import sqlite3
import sys

# Adicionar path do projeto
project_root = Path("B:/_repositorios/skybridge")
sys.path.insert(0, str(project_root))

from src.core.paper.domain.strategies.guardiao_conservador import GuardiaoConservador
from src.core.paper.domain.strategies.signal import DadosMercado, TipoSinal

print('Imports OK')

Imports OK


## 1. Carregar Dados Reais

In [2]:
# Carregar 7d de candles 1m do CSV baixado do Yahoo Finance
csv_path = project_root / "src/core/paper/data/btc_7d_1m.csv"
df = pd.read_csv(csv_path, index_col=0, parse_dates=True)

print(f'Candles: {len(df):,}')
print(f'Range: {df.index[0]} -> {df.index[-1]}')
print(f'Preco: ${df.Low.min():,.2f} - ${df.High.max():,.2f}')
print(f'Swing: ${df.High.max() - df.Low.min():,.2f} ({(df.High.max() - df.Low.min()) / df.Low.min() * 100:.2f}%)')
print(f'\nColunas: {list(df.columns)}')
df.tail()

Candles: 9,685
Range: 2026-05-04 00:00:00+00:00 -> 2026-05-10 21:32:00+00:00
Preco: $78,217.90 - $82,814.22
Swing: $4,596.32 (5.88%)

Colunas: ['Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits']


,Open,High,Low,Close,Volume,Dividends,Stock Splits
Datetime,,,,,,,
2026-05-10 21:28:00+00:00,80744.000000,80795.046875,80744.000000,80795.046875,0,0.0,0.0
2026-05-10 21:29:00+00:00,80793.929688,80829.609375,80793.929688,80829.601562,0,0.0,0.0
2026-05-10 21:30:00+00:00,80829.609375,80860.492188,80829.601562,80859.289062,56799232,0.0,0.0
2026-05-10 21:31:00+00:00,80845.601562,80869.656250,80842.593750,80869.656250,2775040,0.0,0.0
2026-05-10 21:32:00+00:00,80869.656250,80908.320312,80869.656250,80908.320312,1843200,0.0,0.0


## 2. Simular Estratégia com ADX (DI Crossover)

Reproduz a lógica exata do `GuardiaoConservador` sobre os dados históricos.

In [3]:
strategy = GuardiaoConservador(adx_period=14, adx_threshold=Decimal("25"))

closes = df['Close'].values
highs = df['High'].values
lows = df['Low'].values
volumes = df['Volume'].values.astype(int)

# Rodar estratégia candle a candle
signals = []
adx_values = []
plus_di_values = []
minus_di_values = []

min_candles = 28  # adx_period * 2

for i in range(min_candles, len(df)):
    # Janela deslizante
    window_closes = tuple(Decimal(str(v)) for v in closes[:i+1])
    window_highs = tuple(Decimal(str(v)) for v in highs[:i+1])
    window_lows = tuple(Decimal(str(v)) for v in lows[:i+1])
    window_volumes = tuple(int(v) for v in volumes[:i+1])
    
    dados = DadosMercado(
        ticker='BTC-USD',
        preco_atual=Decimal(str(closes[i])),
        historico_precos=window_closes,
        historico_volumes=window_volumes,
        historico_highs=window_highs,
        historico_lows=window_lows,
    )
    
    sinal = strategy.evaluate(dados)
    
    # Registrar indicadores
    ind = strategy._last_indicators
    if ind:
        adx_values.append(float(ind['adx']))
        plus_di_values.append(float(ind['plus_di']))
        minus_di_values.append(float(ind['minus_di']))
    else:
        adx_values.append(0)
        plus_di_values.append(0)
        minus_di_values.append(0)
    
    if sinal is not None:
        signals.append({
            'idx': i,
            'time': df.index[i],
            'tipo': str(sinal.tipo),
            'preco': float(sinal.preco),
            'razao': sinal.razao,
            'take_profit_pct': float(sinal.take_profit_pct),
            'adx': float(ind['adx']) if ind else 0,
            'plus_di': float(ind['plus_di']) if ind else 0,
            'minus_di': float(ind['minus_di']) if ind else 0,
        })

print(f'Sinais gerados: {len(signals)}')
print(f'Compras: {sum(1 for s in signals if s["tipo"] == "TipoSinal.COMPRA")}')
print(f'Vendas: {sum(1 for s in signals if s["tipo"] == "TipoSinal.VENDA")}')

# Dataframe de sinais
df_signals = pd.DataFrame(signals)
if len(df_signals) > 0:
    print(f'\nPrimeiros sinais:')
    print(df_signals[['time', 'tipo', 'preco', 'adx', 'take_profit_pct']].head(10).to_string())

Sinais gerados: 144
Compras: 74
Vendas: 70

Primeiros sinais:
                       time              tipo         preco        adx  take_profit_pct
0 2026-05-04 00:32:00+00:00  TipoSinal.COMPRA  78420.906250  35.060472            0.005
1 2026-05-04 00:37:00+00:00   TipoSinal.VENDA  78334.500000  25.487421            0.004
2 2026-05-04 02:25:00+00:00   TipoSinal.VENDA  79640.156250  39.982948            0.005
3 2026-05-04 02:26:00+00:00  TipoSinal.COMPRA  79746.000000  37.412571            0.005
4 2026-05-04 03:02:00+00:00   TipoSinal.VENDA  80012.632812  41.020744            0.006
5 2026-05-04 03:03:00+00:00  TipoSinal.COMPRA  80107.992188  38.350664            0.005
6 2026-05-04 04:22:00+00:00   TipoSinal.VENDA  80310.507812  28.527536            0.004
7 2026-05-04 07:00:00+00:00   TipoSinal.VENDA  79809.703125  29.918708            0.004
8 2026-05-04 07:01:00+00:00  TipoSinal.COMPRA  79876.703125  27.785961            0.004
9 2026-05-04 07:44:00+00:00  TipoSinal.COMPRA  79852.06250

## 3. Backtest: SL Fixo vs SL Dinâmico

Simula cada par COMPRA->VENDA como um trade, aplicando os dois modelos de SL.

In [6]:
# Modelo de SL dinâmico
def sl_for_adx(adx: float) -> float:
    """Mapeia ADX para SL dinamico (simetrico ao TP)."""
    if adx < 20:
        return 0.0015  # 0.15%
    if adx < 30:
        return 0.0020  # 0.20% (mesmo que fixo atual)
    if adx < 40:
        return 0.0030  # 0.30%
    return 0.0040  # 0.40%

def tp_for_adx(adx: float) -> float:
    """Mapeia ADX para TP dinamico (existente)."""
    if adx < 20:
        return 0.0030
    if adx < 30:
        return 0.0040
    if adx < 40:
        return 0.0050
    return 0.0060

print('Modelos definidos:')
print(f'  SL fixo atual: -0.20%')
print(f'  SL dinamico: ADX<20=0.15%, 20-30=0.20%, 30-40=0.30%, >40=0.40%')
print(f'  TP dinamico: ADX<20=0.30%, 20-30=0.40%, 30-40=0.50%, >40=0.60%')

Modelos definidos:
  SL fixo atual: -0.20%
  SL dinamico: ADX<20=0.15%, 20-30=0.20%, 30-40=0.30%, >40=0.40%
  TP dinamico: ADX<20=0.30%, 20-30=0.40%, 30-40=0.50%, >40=0.60%


In [7]:
QTY = 1  # Quantidade por trade (1 BTC unit)

def run_backtest(df, df_signals, sl_mode='fixed', sl_fixed=0.002):
    """
    Roda backtest de DI crossover.
    
    sl_mode: 'fixed' (SL fixo) ou 'dynamic' (SL por ADX)
    Retorna lista de trades com PnL.
    """
    closes = df['Close'].values
    trades = []
    
    # Filtrar apenas compras (entry) e vendas (exit signal)
    buy_signals = [s for s in df_signals.itertuples() if s.tipo == 'TipoSinal.COMPRA']
    sell_signals = [s for s in df_signals.itertuples() if s.tipo == 'TipoSinal.VENDA']
    
    position = None  # {entry_idx, entry_price, entry_adx, tp_pct}
    
    for i in range(len(df)):
        price = closes[i]
        
        # Checar se ha sinal de compra neste candle
        buy_at = [b for b in buy_signals if b.idx == i]
        sell_at = [s for s in sell_signals if s.idx == i]
        
        # Se tem posicao, checar SL/TP/trailing
        if position is not None:
            entry_price = position['entry_price']
            tp_pct = position['tp_pct']
            adx_at_entry = position['entry_adx']
            
            # SL depende do modo
            if sl_mode == 'fixed':
                sl_pct = sl_fixed
            else:
                sl_pct = sl_for_adx(adx_at_entry)
            
            pnl_pct = (price - entry_price) / entry_price
            
            # Trailing stop (0.20% activation, 0.15% distance)
            if position.get('trailing_activated'):
                position['trailing_pico'] = max(position.get('trailing_pico', price), price)
                trailing_stop = position['trailing_pico'] * (1 - 0.0015)
                trailing_stop = max(trailing_stop, entry_price)  # breakeven floor
                if price <= trailing_stop:
                    pnl_dollar = (price - entry_price) * QTY
                    trades.append({
                        'entry_time': position['entry_time'],
                        'exit_time': df.index[i],
                        'entry_price': entry_price,
                        'exit_price': price,
                        'pnl_pct': pnl_pct,
                        'pnl_dollar': pnl_dollar,
                        'reason': 'trailing_stop',
                        'adx_at_entry': adx_at_entry,
                        'sl_mode': sl_mode,
                        'sl_pct': sl_pct,
                        'tp_pct': tp_pct,
                    })
                    position = None
                    continue
            else:
                # Check trailing activation (0.20% profit)
                if pnl_pct >= 0.002:
                    position['trailing_activated'] = True
                    position['trailing_pico'] = price
            
            # Stop Loss
            if pnl_pct <= -sl_pct:
                pnl_dollar = (price - entry_price) * QTY
                trades.append({
                    'entry_time': position['entry_time'],
                    'exit_time': df.index[i],
                    'entry_price': entry_price,
                    'exit_price': price,
                    'pnl_pct': pnl_pct,
                    'pnl_dollar': pnl_dollar,
                    'reason': 'stop_loss',
                    'adx_at_entry': adx_at_entry,
                    'sl_mode': sl_mode,
                    'sl_pct': sl_pct,
                    'tp_pct': tp_pct,
                })
                position = None
                continue
            
            # Take Profit
            if pnl_pct >= tp_pct:
                pnl_dollar = (price - entry_price) * QTY
                trades.append({
                    'entry_time': position['entry_time'],
                    'exit_time': df.index[i],
                    'entry_price': entry_price,
                    'exit_price': price,
                    'pnl_pct': pnl_pct,
                    'pnl_dollar': pnl_dollar,
                    'reason': 'take_profit',
                    'adx_at_entry': adx_at_entry,
                    'sl_mode': sl_mode,
                    'sl_pct': sl_pct,
                    'tp_pct': tp_pct,
                })
                position = None
                continue
            
            # Sinal de venda do crossover
            if sell_at:
                pnl_dollar = (price - entry_price) * QTY
                trades.append({
                    'entry_time': position['entry_time'],
                    'exit_time': df.index[i],
                    'entry_price': entry_price,
                    'exit_price': price,
                    'pnl_pct': pnl_pct,
                    'pnl_dollar': pnl_dollar,
                    'reason': 'di_crossover',
                    'adx_at_entry': adx_at_entry,
                    'sl_mode': sl_mode,
                    'sl_pct': sl_pct,
                    'tp_pct': tp_pct,
                })
                position = None
                continue
        
        # Se nao tem posicao e tem sinal de compra
        if position is None and buy_at:
            b = buy_at[0]
            adx_val = b.adx
            position = {
                'entry_idx': i,
                'entry_price': price,
                'entry_time': df.index[i],
                'entry_adx': adx_val,
                'tp_pct': tp_for_adx(adx_val),
            }
    
    return trades

print(f'Funcao de backtest definida (QTY={QTY})')

Funcao de backtest definida (QTY=1)


In [8]:
# Rodar os dois backtests
if len(df_signals) > 0:
    trades_fixed = run_backtest(df, df_signals, sl_mode='fixed', sl_fixed=0.002)
    trades_dynamic = run_backtest(df, df_signals, sl_mode='dynamic')
    
    df_fixed = pd.DataFrame(trades_fixed)
    df_dynamic = pd.DataFrame(trades_dynamic)
    
    print(f'Trades SL Fixo: {len(df_fixed)}')
    print(f'Trades SL Dinamico: {len(df_dynamic)}')
else:
    print('Sem sinais para backtest')

Trades SL Fixo: 69
Trades SL Dinamico: 69


## 4. KPIs Comparativos

In [9]:
def calc_kpis(trades_df, label=''):
    if len(trades_df) == 0:
        print(f'{label}: SEM TRADES')
        return {}
    
    wins = trades_df[trades_df['pnl_dollar'] > 0]
    losses = trades_df[trades_df['pnl_dollar'] <= 0]
    
    total_pnl = trades_df['pnl_dollar'].sum()
    n_wins = len(wins)
    n_losses = len(losses)
    win_rate = n_wins / len(trades_df) * 100
    
    avg_win = wins['pnl_dollar'].mean() if len(wins) > 0 else 0
    avg_loss = abs(losses['pnl_dollar'].mean()) if len(losses) > 0 else 0
    rr_ratio = avg_win / avg_loss if avg_loss > 0 else float('inf')
    
    gross_profit = wins['pnl_dollar'].sum() if len(wins) > 0 else 0
    gross_loss = abs(losses['pnl_dollar'].sum()) if len(losses) > 0 else 0
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')
    
    # Sharpe proxy (mean/std of returns)
    returns = trades_df['pnl_dollar'].values
    sharpe = np.mean(returns) / np.std(returns) if np.std(returns) > 0 else 0
    
    # Max drawdown
    cumulative = np.cumsum(returns)
    peak = np.maximum.accumulate(cumulative)
    drawdown = cumulative - peak
    max_dd = drawdown.min()
    
    kpis = {
        'label': label,
        'n_trades': len(trades_df),
        'total_pnl': total_pnl,
        'win_rate': win_rate,
        'avg_win': avg_win,
        'avg_loss': avg_loss,
        'rr_ratio': rr_ratio,
        'profit_factor': profit_factor,
        'sharpe_proxy': sharpe,
        'max_drawdown': max_dd,
    }
    
    print(f'=== {label} ===')
    print(f'  Trades: {kpis["n_trades"]}')
    print(f'  PnL Total: ${total_pnl:,.2f}')
    print(f'  Win Rate: {win_rate:.1f}% ({n_wins}W / {n_losses}L)')
    print(f'  Avg Win: ${avg_win:,.2f}')
    print(f'  Avg Loss: ${avg_loss:,.2f}')
    print(f'  R:R Ratio: {rr_ratio:.2f}:1')
    print(f'  Profit Factor: {profit_factor:.2f}')
    print(f'  Sharpe Proxy: {sharpe:.2f}')
    print(f'  Max Drawdown: ${max_dd:,.2f}')
    print()
    return kpis

if len(df_fixed) > 0 and len(df_dynamic) > 0:
    kpis_fixed = calc_kpis(df_fixed, 'SL FIXO (-0.20%)')
    kpis_dynamic = calc_kpis(df_dynamic, 'SL DINAMICO (ADX)')
else:
    print('Rodar celulas anteriores primeiro')

=== SL FIXO (-0.20%) ===
  Trades: 69
  PnL Total: $-573.95
  Win Rate: 44.9% (31W / 38L)
  Avg Win: $135.61
  Avg Loss: $125.73
  R:R Ratio: 1.08:1
  Profit Factor: 0.88
  Sharpe Proxy: -0.05
  Max Drawdown: $-1,598.80

=== SL DINAMICO (ADX) ===
  Trades: 69
  PnL Total: $-441.38
  Win Rate: 46.4% (32W / 37L)
  Avg Win: $133.44
  Avg Loss: $127.34
  R:R Ratio: 1.05:1
  Profit Factor: 0.91
  Sharpe Proxy: -0.04
  Max Drawdown: $-1,408.01



## 5. Análise por Razão de Saída

In [10]:
def breakdown_by_reason(trades_df, label=''):
    if len(trades_df) == 0:
        return
    print(f'=== Breakdown: {label} ===')
    for reason in trades_df['reason'].unique():
        subset = trades_df[trades_df['reason'] == reason]
        n = len(subset)
        pnl = subset['pnl_dollar'].sum()
        avg = subset['pnl_dollar'].mean()
        wins = len(subset[subset['pnl_dollar'] > 0])
        print(f'  {reason:20s}: {n:3d} trades | PnL ${pnl:>10,.2f} | avg ${avg:>8,.2f} | WR {wins/n*100:.0f}%')
    print()

if len(df_fixed) > 0:
    breakdown_by_reason(df_fixed, 'SL FIXO')
if len(df_dynamic) > 0:
    breakdown_by_reason(df_dynamic, 'SL DINAMICO')

=== Breakdown: SL FIXO ===
  di_crossover        :  31 trades | PnL $   -236.46 | avg $   -7.63 | WR 42%
  take_profit         :   6 trades | PnL $  2,289.92 | avg $  381.65 | WR 100%
  trailing_stop       :  12 trades | PnL $  1,091.43 | avg $   90.95 | WR 100%
  stop_loss           :  20 trades | PnL $ -3,718.84 | avg $ -185.94 | WR 0%

=== Breakdown: SL DINAMICO ===
  di_crossover        :  32 trades | PnL $   -170.34 | avg $   -5.32 | WR 44%
  take_profit         :   6 trades | PnL $  2,289.92 | avg $  381.65 | WR 100%
  trailing_stop       :  12 trades | PnL $  1,091.43 | avg $   90.95 | WR 100%
  stop_loss           :  19 trades | PnL $ -3,652.38 | avg $ -192.23 | WR 0%



## 6. Distribuição de PnL por ADX Region

Verifica se o SL dinâmico realmente ajuda em cada regime de ADX.

In [11]:
def adx_bucket(adx):
    if adx < 20: return '<20'
    if adx < 30: return '20-30'
    if adx < 40: return '30-40'
    return '>40'

if len(df_fixed) > 0 and len(df_dynamic) > 0:
    df_fixed['adx_bucket'] = df_fixed['adx_at_entry'].apply(adx_bucket)
    df_dynamic['adx_bucket'] = df_dynamic['adx_at_entry'].apply(adx_bucket)
    
    print('PnL por ADX bucket:')
    print(f'{"Bucket":>8} | {"SL Fixo PnL":>14} | {"SL Din PnL":>14} | {"Fixo Trades":>11} | {"Din Trades":>11}')
    print('-' * 70)
    
    for bucket in ['<20', '20-30', '30-40', '>40']:
        f = df_fixed[df_fixed['adx_bucket'] == bucket]
        d = df_dynamic[df_dynamic['adx_bucket'] == bucket]
        pnl_f = f['pnl_dollar'].sum() if len(f) > 0 else 0
        pnl_d = d['pnl_dollar'].sum() if len(d) > 0 else 0
        n_f = len(f)
        n_d = len(d)
        print(f'{bucket:>8} | ${pnl_f:>12,.2f} | ${pnl_d:>12,.2f} | {n_f:>11d} | {n_d:>11d}')
    
    print(f'{"TOTAL":>8} | ${df_fixed["pnl_dollar"].sum():>12,.2f} | ${df_dynamic["pnl_dollar"].sum():>12,.2f} | {len(df_fixed):>11d} | {len(df_dynamic):>11d}')

PnL por ADX bucket:
  Bucket |    SL Fixo PnL |     SL Din PnL | Fixo Trades |  Din Trades
----------------------------------------------------------------------
     <20 | $        0.00 | $        0.00 |           0 |           0
   20-30 | $     -518.24 | $     -518.24 |          44 |          44
   30-40 | $      -37.66 | $       94.91 |          22 |          22
     >40 | $      -18.04 | $      -18.04 |           3 |           3
   TOTAL | $     -573.95 | $     -441.38 |          69 |          69


## 7. Comparação Visual: SL Fixo vs Dinâmico

In [12]:
# Tabela resumo lado a lado
if 'kpis_fixed' in dir() and 'kpis_dynamic' in dir():
    print(f'{"KPI":>20} | {"SL Fixo":>16} | {"SL Dinamico":>16} | {"Delta":>12}')
    print('=' * 75)
    
    for key in ['n_trades', 'total_pnl', 'win_rate', 'avg_win', 'avg_loss', 
                'rr_ratio', 'profit_factor', 'sharpe_proxy', 'max_drawdown']:
        f_val = kpis_fixed.get(key, 0)
        d_val = kpis_dynamic.get(key, 0)
        
        if key == 'n_trades':
            delta = d_val - f_val
            print(f'{key:>20} | {f_val:>16d} | {d_val:>16d} | {delta:>+12d}')
        elif key in ('win_rate',):
            delta = d_val - f_val
            print(f'{key:>20} | {f_val:>15.1f}% | {d_val:>15.1f}% | {delta:>+11.1f}pp')
        elif key == 'rr_ratio':
            delta = d_val - f_val
            print(f'{key:>20} | {f_val:>14.2f}:1 | {d_val:>14.2f}:1 | {delta:>+12.2f}')
        else:
            delta = d_val - f_val
            fmt = '.2f'
            print(f'{key:>20} | {f_val:>16,.2f} | {d_val:>16,.2f} | {delta:>+12,.2f}')

                 KPI |          SL Fixo |      SL Dinamico |        Delta
            n_trades |               69 |               69 |           +0
           total_pnl |          -573.95 |          -441.38 |      +132.57
            win_rate |            44.9% |            46.4% |        +1.4pp
             avg_win |           135.61 |           133.44 |        -2.17
            avg_loss |           125.73 |           127.34 |        +1.60
            rr_ratio |           1.08:1 |           1.05:1 |        -0.03
       profit_factor |             0.88 |             0.91 |        +0.03
        sharpe_proxy |            -0.05 |            -0.04 |        +0.01
        max_drawdown |        -1,598.80 |        -1,408.01 |      +190.80


## 8. Stress Test: Sensibilidade dos Thresholds

Testa diferentes combinações de SL/TP para encontrar o ponto ótimo.

In [13]:
# Grid search de SL fixo vs dynamic variants
if len(df_signals) > 0:
    results = []
    
    # SL fixo: testar de 0.10% a 0.50%
    for sl_pct in [0.0010, 0.0015, 0.0020, 0.0025, 0.0030, 0.0040, 0.0050]:
        trades = run_backtest(df, df_signals, sl_mode='fixed', sl_fixed=sl_pct)
        if trades:
            tdf = pd.DataFrame(trades)
            wins = tdf[tdf['pnl_dollar'] > 0]
            losses = tdf[tdf['pnl_dollar'] <= 0]
            pnl = tdf['pnl_dollar'].sum()
            wr = len(wins) / len(tdf) * 100 if len(tdf) > 0 else 0
            pf = wins['pnl_dollar'].sum() / abs(losses['pnl_dollar'].sum()) if len(losses) > 0 and losses['pnl_dollar'].sum() != 0 else float('inf')
            results.append({
                'mode': f'fixed_{sl_pct:.2%}',
                'sl_pct': sl_pct,
                'n_trades': len(tdf),
                'pnl': pnl,
                'win_rate': wr,
                'profit_factor': pf,
            })
    
    # SL dinamico (proposto)
    trades_dyn = run_backtest(df, df_signals, sl_mode='dynamic')
    if trades_dyn:
        tdf = pd.DataFrame(trades_dyn)
        wins = tdf[tdf['pnl_dollar'] > 0]
        losses = tdf[tdf['pnl_dollar'] <= 0]
        pnl = tdf['pnl_dollar'].sum()
        wr = len(wins) / len(tdf) * 100
        pf = wins['pnl_dollar'].sum() / abs(losses['pnl_dollar'].sum()) if len(losses) > 0 and losses['pnl_dollar'].sum() != 0 else float('inf')
        results.append({
            'mode': 'DYNAMIC_ADX',
            'sl_pct': 0,
            'n_trades': len(tdf),
            'pnl': pnl,
            'win_rate': wr,
            'profit_factor': pf,
        })
    
    df_results = pd.DataFrame(results).sort_values('pnl', ascending=False)
    print('Ranking de estrategias por PnL:')
    print(df_results[['mode', 'n_trades', 'pnl', 'win_rate', 'profit_factor']].to_string(index=False))
else:
    print('Sem dados suficientes')

Ranking de estrategias por PnL:
       mode  n_trades         pnl  win_rate  profit_factor
fixed_0.50%        63 1346.132812 57.142857       1.380591
fixed_0.40%        64 1009.984375 54.687500       1.263831
fixed_0.30%        66  354.875000 51.515152       1.084936
fixed_0.10%        74  -62.789062 39.189189       0.983882
fixed_0.15%        73 -351.671875 42.465753       0.923590
DYNAMIC_ADX        69 -441.375000 46.376812       0.906318
fixed_0.20%        69 -573.945312 44.927536       0.879875
fixed_0.25%        66 -750.671875 45.454545       0.845666


## 9. PnL Cumulativo ao Longo do Tempo

In [14]:
if len(df_fixed) > 0 and len(df_dynamic) > 0:
    cum_fixed = df_fixed['pnl_dollar'].cumsum()
    cum_dynamic = df_dynamic['pnl_dollar'].cumsum()
    
    print('PnL Cumulativo (SL Fixo):')
    for i, val in enumerate(cum_fixed):
        if i % 5 == 0 or i == len(cum_fixed) - 1:
            trade = df_fixed.iloc[i]
            print(f'  Trade #{i+1:3d} | {str(trade["exit_time"])[:19]} | ${val:>10,.2f} | {trade["reason"]}')
    
    print(f'\nPnL Cumulativo (SL Dinamico):')
    for i, val in enumerate(cum_dynamic):
        if i % 5 == 0 or i == len(cum_dynamic) - 1:
            trade = df_dynamic.iloc[i]
            print(f'  Trade #{i+1:3d} | {str(trade["exit_time"])[:19]} | ${val:>10,.2f} | {trade["reason"]}')

PnL Cumulativo (SL Fixo):
  Trade #  1 | 2026-05-04 00:37:00 | $    -86.41 | di_crossover
  Trade #  6 | 2026-05-04 10:54:00 | $    245.75 | trailing_stop
  Trade # 11 | 2026-05-04 22:34:00 | $   -128.80 | stop_loss
  Trade # 16 | 2026-05-05 10:53:00 | $    -34.59 | di_crossover
  Trade # 21 | 2026-05-05 17:31:00 | $    481.56 | trailing_stop
  Trade # 26 | 2026-05-06 04:51:00 | $    227.86 | di_crossover
  Trade # 31 | 2026-05-06 17:46:00 | $    586.80 | di_crossover
  Trade # 36 | 2026-05-07 14:16:00 | $    183.22 | di_crossover
  Trade # 41 | 2026-05-08 00:27:00 | $   -169.62 | di_crossover
  Trade # 46 | 2026-05-08 09:40:00 | $    -74.31 | stop_loss
  Trade # 51 | 2026-05-08 19:02:00 | $   -659.63 | stop_loss
  Trade # 56 | 2026-05-09 05:02:00 | $   -758.19 | stop_loss
  Trade # 61 | 2026-05-10 07:14:00 | $   -389.98 | di_crossover
  Trade # 66 | 2026-05-10 16:21:00 | $   -348.70 | di_crossover
  Trade # 69 | 2026-05-10 21:11:00 | $   -573.95 | stop_loss

PnL Cumulativo (SL Dinamic

## 10. Conclusão e Recomendação

In [15]:
if 'kpis_fixed' in dir() and 'kpis_dynamic' in dir():
    pnl_delta = kpis_dynamic['total_pnl'] - kpis_fixed['total_pnl']
    pnl_pct_delta = (pnl_delta / abs(kpis_fixed['total_pnl']) * 100) if kpis_fixed['total_pnl'] != 0 else 0
    wr_delta = kpis_dynamic['win_rate'] - kpis_fixed['win_rate']
    sharpe_delta = kpis_dynamic['sharpe_proxy'] - kpis_fixed['sharpe_proxy']
    
    print('RESUMO EXECUTIVO')
    print('=' * 50)
    print(f'PnL: ${kpis_fixed["total_pnl"]:,.2f} -> ${kpis_dynamic["total_pnl"]:,.2f} ({pnl_delta:+,.2f} / {pnl_pct_delta:+.0f}%)')
    print(f'Win Rate: {kpis_fixed["win_rate"]:.1f}% -> {kpis_dynamic["win_rate"]:.1f}% ({wr_delta:+.1f}pp)')
    print(f'Sharpe: {kpis_fixed["sharpe_proxy"]:.2f} -> {kpis_dynamic["sharpe_proxy"]:.2f} ({sharpe_delta:+.2f})')
    print(f'Profit Factor: {kpis_fixed["profit_factor"]:.2f} -> {kpis_dynamic["profit_factor"]:.2f}')
    print()
    
    if pnl_delta > 0 and sharpe_delta > 0:
        print('RECOMENDACAO: IMPLEMENTAR SL DINAMICO')
        print('Resultados positivos em PnL e Sharpe confirmam a hipotese.')
    elif pnl_delta > 0:
        print('RECOMENDACAO: CAUTELA - PnL melhorou mas Sharpe piorou')
        print('Pode haver mais risco por trade.')
    else:
        print('RECOMENDACAO: MANTER SL FIXO')
        print('SL dinamico nao mostrou melhoria suficiente.')
    print()
    print('Tabela final:')
    print('| ADX    | SL Dinamico | TP Dinamico |')
    print('|--------|------------|-------------|')
    print('| < 20   | 0.15%      | 0.30%       |')
    print('| 20-30  | 0.20%      | 0.40%       |')
    print('| 30-40  | 0.30%      | 0.50%       |')
    print('| > 40   | 0.40%      | 0.60%       |')

RESUMO EXECUTIVO
PnL: $-573.95 -> $-441.38 (+132.57 / +23%)
Win Rate: 44.9% -> 46.4% (+1.4pp)
Sharpe: -0.05 -> -0.04 (+0.01)
Profit Factor: 0.88 -> 0.91

RECOMENDACAO: IMPLEMENTAR SL DINAMICO
Resultados positivos em PnL e Sharpe confirmam a hipotese.

Tabela final:
| ADX    | SL Dinamico | TP Dinamico |
|--------|------------|-------------|
| < 20   | 0.15%      | 0.30%       |
| 20-30  | 0.20%      | 0.40%       |
| 30-40  | 0.30%      | 0.50%       |
| > 40   | 0.40%      | 0.60%       |
